# Day 26 — Spark Connect & Unity Catalog

**What you will learn:**
- Spark Connect: decoupled client-server architecture
- Unity Catalog: 3-level namespace, governance model, fine-grained access control
- Catalog API: listing catalogs, databases, tables
- Managed vs external tables in Unity Catalog
- Column-level and row-level security concepts

**Exam relevance:** Spark Connect (5%) and Unity Catalog concepts appear in the exam — know the 3-level namespace and what Unity Catalog governs.

## 1. Spark Connect Architecture

Traditional Spark driver-executor model:
```
Client (Python) ←→ Driver (JVM) ←→ Executors
    (same process or tight coupling)
```

Spark Connect (Spark 3.4+):
```
Client (Python/Go/Scala) ──gRPC──► Spark Connect Server (JVM) ←→ Executors
    (thin client, no JVM needed)
```

**Key benefits:**
- Client crashes don't kill the Spark session
- Any language can talk to Spark via gRPC (Python, Go, R, Scala)
- IDE debugging of Spark logic without a local cluster
- Databricks Serverless runs on Spark Connect

**API change:**
```python
# Old: SparkSession.builder.getOrCreate()
# Spark Connect:
from pyspark.sql import SparkSession
spark = SparkSession.builder.remote("sc://localhost").getOrCreate()
```

> **Exam tip:** Spark Connect does NOT support direct RDD access (`spark.sparkContext.*`). Only the DataFrame/SQL API is available — this is why all course notebooks wrap RDD calls in try/except.

In [ ]:
# Detect whether we're on Spark Connect (serverless)
try:
    sc = spark.sparkContext
    print(f"Classic Spark — sparkContext available: {sc.version}")
except Exception as e:
    print(f"Spark Connect (serverless) — no sparkContext: {e}")

# DataFrame API works identically in both modes
spark.range(5).show()

## 2. Unity Catalog — 3-Level Namespace

```
catalog
└── schema (database)
    └── table / view / function / volume
```

Fully qualified name:
```sql
catalog_name.schema_name.table_name
```

**Default catalog:** `spark_catalog` (Hive Metastore compatibility)

**What Unity Catalog governs:**
- Tables and views (Delta, Parquet, CSV, ...)
- Volumes (unstructured file storage)
- ML models, functions, notebooks
- Fine-grained access control (column masking, row filtering)

In [ ]:
# Catalog API — works in Hive Metastore (classic) and Unity Catalog

# List available catalogs
print("=== Catalogs ===")
spark.catalog.listDatabases()  # returns list of Database objects

# Current catalog and database
try:
    print(f"Current catalog:  {spark.catalog.currentCatalog()}")
except Exception:
    print("currentCatalog() not available (older Spark version)")

print(f"Current database: {spark.catalog.currentDatabase()}")

In [ ]:
# Create a database and table via SQL
spark.sql("CREATE DATABASE IF NOT EXISTS demo_db")
spark.sql("USE demo_db")

# Managed table — Spark controls the lifecycle and storage location
spark.sql("""
    CREATE TABLE IF NOT EXISTS employees (
        id     INT,
        name   STRING,
        dept   STRING,
        salary INT
    ) USING delta
""")

spark.sql("""
    INSERT INTO employees VALUES
    (1, 'Alice', 'Engineering', 95000),
    (2, 'Bob',   'Marketing',   72000)
""")

spark.sql("SELECT * FROM employees").show()

In [ ]:
# List tables in current database
print("=== Tables in demo_db ===")
for t in spark.catalog.listTables("demo_db"):
    print(f"  {t.name} ({t.tableType}) — isTemporary={t.isTemporary}")

## 3. Managed vs External Tables

| Property | Managed | External |
|---|---|---|
| Data location | Managed by Spark/Unity Catalog | You specify the path |
| DROP TABLE | Deletes metadata AND data | Deletes metadata only |
| Use case | Typical Delta tables | Pre-existing data in S3/ADLS |

In [ ]:
import tempfile
EXTERNAL_PATH = tempfile.mkdtemp(prefix="ext_table_")

# External table — data at a specific path
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS ext_orders (
        order_id STRING,
        amount   DOUBLE
    ) USING delta
    LOCATION '{EXTERNAL_PATH}'
""")

spark.sql("INSERT INTO ext_orders VALUES ('ORD-001', 99.5), ('ORD-002', 200.0)")

print("=== DESCRIBE DETAIL ===")
spark.sql("DESCRIBE DETAIL ext_orders").select("name", "location", "tableType").show(truncate=False)

# DROP external table — only removes metadata, files remain
spark.sql("DROP TABLE ext_orders")
print(f"Files still exist at: {EXTERNAL_PATH}")
import os
print(os.listdir(EXTERNAL_PATH))

## 4. Unity Catalog Access Control (Conceptual)

Unity Catalog uses `GRANT` / `REVOKE` at catalog, schema, table, column, or row level.

```sql
-- Catalog level
GRANT USE CATALOG ON CATALOG main TO `data-team`;

-- Schema level
GRANT USE SCHEMA ON SCHEMA main.sales TO `analysts`;

-- Table level
GRANT SELECT ON TABLE main.sales.orders TO `analysts`;

-- Column masking (Databricks)
CREATE FUNCTION mask_email(email STRING)
RETURN CASE WHEN is_account_group_member('pii-allowed') THEN email
            ELSE '***@***.***' END;

ALTER TABLE customers ALTER COLUMN email
    SET MASK mask_email;

-- Row filter (Databricks)
CREATE FUNCTION only_my_region(region STRING)
RETURN region = current_user();

ALTER TABLE orders SET ROW FILTER only_my_region ON (region);
```

## 5. Catalog API Reference

In [ ]:
# Table existence check
print("employees exists:", spark.catalog.tableExists("demo_db.employees"))
print("nonexistent exists:", spark.catalog.tableExists("demo_db.nonexistent"))

# Describe columns
print("=== Columns ===")
for c in spark.catalog.listColumns("employees", "demo_db"):
    print(f"  {c.name}: {c.dataType} (nullable={c.nullable})")

# Refresh table metadata (after external changes)
spark.catalog.refreshTable("demo_db.employees")
print("Table refreshed.")

---

## Day 26 Checklist

- [ ] Know Spark Connect: thin gRPC client, no JVM, no sparkContext access
- [ ] Know the 3-level Unity Catalog namespace: `catalog.schema.table`
- [ ] Know what Unity Catalog governs: tables, volumes, models, functions
- [ ] Created managed and external Delta tables — know DROP TABLE difference
- [ ] Used `spark.catalog.*` API to list tables, check existence, describe columns
- [ ] Know Unity Catalog GRANT hierarchy: catalog → schema → table → column/row

**Next:** Day 27 — Final Project: Complete Data Engineering Pipeline